Import des bibliothèques

In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PooledOLS
from linearmodels.panel import PanelOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

Import des données

In [3]:
data = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\données_caracteristiques_communes.csv", index_col=0)
data['codecommune'] = data['codecommune'].astype(str).str.zfill(5)

# correction de la variable pbac
data['pbac'] = (data['pbac'] - data['psup']).clip(lower=0)

data.head()

C:\Users\yancr\AppData\Local\Temp\ipykernel_29584\184722120.py:1: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\données_caracteristiques_communes.csv", index_col=0)


,codecommune,Annee,pop,propf,prop014,prop1539,prop4059,prop60p,petranger,pcapi,...,pbac,psup,revmoy,vote_EG_pres,abstention_pres,vote_RN_pres,vote_EG_leg,abstention_leg,vote_RN_leg,typologie_urbaine
0,01001,1995,662,0.471756,0.268702,0.280916,0.258015,0.192366,0.016320,0.294118,...,0.044811,0.073113,12110.340,0.076056,0.201285,0.225352,NaN,NaN,NaN,Rural non périurbain
1,01001,1996,678,0.468657,0.270149,0.270149,0.265672,0.194030,0.017366,0.297521,...,0.045872,0.075688,11530.236,NaN,NaN,NaN,NaN,NaN,NaN,Rural non périurbain
2,01001,1997,695,0.467930,0.272595,0.259475,0.271137,0.196793,0.016997,0.298387,...,0.046875,0.080357,12207.807,NaN,NaN,NaN,0.098639,0.337607,0.20068,Rural non périurbain
3,01001,1998,711,0.465714,0.272857,0.250000,0.278571,0.198571,0.018006,0.305882,...,0.049892,0.082429,12892.058,NaN,NaN,NaN,NaN,NaN,NaN,Rural non périurbain
4,01001,1999,728,0.463687,0.273743,0.240223,0.284916,0.201117,0.018945,0.307692,...,0.050847,0.084746,12839.338,NaN,NaN,NaN,NaN,NaN,NaN,Rural non périurbain


# Elections Présidentielles

## Première partie : Evaluer l'évolution du vote pour le Rassemblement National entre 1995 et 2022 selon le type de communes

### Modélisation

Modèle naïf utilisé : 

$$VoteRN_{it} = \alpha_i + \gamma_t + \sum_{\tau \in \{2002, 2007, 2012, 2017, 2022\}} \beta_{\tau} (Typologie_i \times \mathbb{1}_{t=\tau}) + \epsilon_{it}$$

Modèle avec contrôles : 

$$VoteRN_{it} = \alpha_i + \gamma_t + \sum_{\tau \in \{2002, 2007, 2012, 2017, 2022\}} \widetilde{\beta}_{\tau} (Typologie_i \times \mathbb{1}_{t=\tau}) + \delta X_{it} + \eta_{it}$$

### Estimation du modèle naif

Selection des variables

In [4]:
df_modele_naif = data.dropna(subset=['vote_RN_pres', 'typologie_urbaine', 'Annee', 'codecommune']).copy()

Création des interactions typologie x annee

In [5]:
# Liste des années (en excluant 1995 qui sera notre référence)
annees_elections = sorted([y for y in df_modele_naif['Annee'].unique() if y > 1995])

# Liste des typologies (en excluant 'Urbain dense' qui est la référence)
typologies_commune = [t for t in df_modele_naif['typologie_urbaine'].unique() if t != 'Urbain dense']

# On crée une colonne pour chaque (Typologie x Année)
interaction_cols = []

for typologie in typologies_commune:
    for annee in annees_elections:
        col_name = f"inter_{typologie.replace(' ', '_')}_{annee}"
        # La variable vaut 1 si la commune appartient à la typo ET que c'est l'année T, sinon 0
        df_modele_naif[col_name] = ((df_modele_naif['typologie_urbaine'] == typologie) & (df_modele_naif['Annee'] == annee)).astype(int)
        interaction_cols.append(col_name)

Passage des données en panel

In [6]:
df_modele_naif = df_modele_naif.set_index(['codecommune', 'Annee'])

Estimation du modèle naïf

In [7]:
formula_inter = 'vote_RN_pres ~ 1 + ' + ' + '.join(interaction_cols) + ' + EntityEffects + TimeEffects'

mod_inter = PanelOLS.from_formula(formula_inter, data=df_modele_naif, drop_absorbed=True)
res_inter = mod_inter.fit(cov_type='clustered', cluster_entity=True)

print("=== MODÈLE : ÉVOLUTION PAR RAPPORT À 1995 (RÉFÉRENCE : URBAIN DENSE) ===")
print(res_inter.summary)

=== MODÈLE : ÉVOLUTION PAR RAPPORT À 1995 (RÉFÉRENCE : URBAIN DENSE) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:           vote_RN_pres   R-squared:                        0.0509
Estimator:                   PanelOLS   R-squared (Between):             -0.0329
No. Observations:              208975   R-squared (Within):               0.5307
Date:                Wed, Apr 22 2026   R-squared (Overall):              0.2934
Time:                        16:21:36   Log-likelihood                 3.699e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      622.47
Entities:                       34850   P-value                           0.0000
Avg Obs:                       5.9964   Distribution:               F(15,174105)
Min Obs:                       2.0000                                           
Max Obs:                       6.000

### Estimation du modèle avec controles

Selection des variables. Pour chaque variable qualitative, on enleve une des catégorie qui deviendra la catégorie de référence.

In [8]:
df_modele_controles = data.dropna(subset=['vote_RN_pres', 'typologie_urbaine', 'Annee', 'codecommune','pop', 'propf', 'prop1539',
'prop4059', 'prop60p', 'petranger', 'pouem', 'paind', 'pchom', 'pbac', 'psup', 'revmoy']).copy()

Création des interactions typologie x annee

In [9]:
# Liste des années (en excluant 1995 qui sera notre référence)
annees_elections = sorted([y for y in df_modele_controles['Annee'].unique() if y > 1995])

# Liste des typologies (en excluant 'Urbain dense' qui est la référence)
typologies_commune = [t for t in df_modele_controles['typologie_urbaine'].unique() if t != 'Urbain dense']

# On crée une colonne pour chaque (Typologie x Année)
interaction_cols = []

for typologie in typologies_commune:
    for annee in annees_elections:
        col_name = f"inter_{typologie.replace(' ', '_')}_{annee}"
        # La variable vaut 1 si la commune appartient à la typo ET que c'est l'année T, sinon 0
        df_modele_controles[col_name] = ((df_modele_controles['typologie_urbaine'] == typologie) & (df_modele_controles['Annee'] == annee)).astype(int)
        interaction_cols.append(col_name)

Passage des données en panel

In [10]:
df_modele_controles = df_modele_controles.set_index(['codecommune', 'Annee'])

Estimation du modèle avec controles

In [11]:
# On définie les controles
controls = ['pop', 'propf', 'prop1539', 'prop4059', 'prop60p', 'petranger', 'pchom', 'pouem', 'paind', 'pbac', 'psup', 'revmoy']

formula_inter = 'vote_RN_pres ~ 1 + ' + ' + '.join(interaction_cols)+ ' + ' + ' + '.join(controls) + ' + EntityEffects + TimeEffects'

mod_inter = PanelOLS.from_formula(formula_inter, data=df_modele_controles, drop_absorbed=True)
res_inter = mod_inter.fit(cov_type='clustered', cluster_entity=True)

print("=== MODÈLE : ÉVOLUTION PAR RAPPORT À 1995 (RÉFÉRENCE : URBAIN DENSE) ===")
print(res_inter.summary)

=== MODÈLE : ÉVOLUTION PAR RAPPORT À 1995 (RÉFÉRENCE : URBAIN DENSE) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:           vote_RN_pres   R-squared:                        0.0707
Estimator:                   PanelOLS   R-squared (Between):             -0.2016
No. Observations:              206016   R-squared (Within):               0.4945
Date:                Wed, Apr 22 2026   R-squared (Overall):              0.2000
Time:                        16:21:41   Log-likelihood                 3.708e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      482.74
Entities:                       34738   P-value                           0.0000
Avg Obs:                       5.9306   Distribution:               F(27,171246)
Min Obs:                       1.0000                                           
Max Obs:                       6.000

## Deuxième partie : Estimation de l'évolution du vote pour le rassemblement national au sein de chaque typologie de communes

### Modélisation

Modèle naïf utilisé : 

$$VoteRN_{it} = \alpha_i + \sum_{\tau \in \{2002, 2007, 2012, 2017, 2022\}} \beta_{\tau} (\mathbb{1}_{t=\tau}) + \epsilon_{it}$$

Modèle avec contrôles : 

$$VoteRN_{it} = \alpha_i + \sum_{\tau \in \{2002, 2007, 2012, 2017, 2022\}} \widetilde{\beta}_{\tau} (\mathbb{1}_{t=\tau}) + \delta X_{it} + \eta_{it}$$

## Deuxième partie : Estimation de l'évolution du vote pour le rassemblement national au sein de chaque typologie de communes

### Communes rurales non périurbaines

#### Modèle naïf

On filtre les données sur la typologie de communes

In [42]:
df_typologie = data[data['typologie_urbaine'] == 'Rural non périurbain'].copy()


Création des intéractions sur les années d'élections

In [43]:
# On crée une colonne pour chaque année d'élection sauf la première
annees_dummies = []
for annee in sorted(df_typologie['Annee'].unique()):
    if annee in [2002, 2007, 2012, 2017, 2022] :
        col_name = f'annee_{annee}'
        df_typologie[col_name] = (df_typologie['Annee'] == annee).astype(int)
        annees_dummies.append(col_name)

Création de la formule du modèle

In [44]:
# Note : On ne met PAS de 'TimeEffects' ici car les dummies années jouent exactement ce rôle.
# On garde les 'EntityEffects' pour voir l'évolution interne à chaque commune.
df_typologie = df_typologie.set_index(['codecommune', 'Annee'])

formula_typologie = (
    'vote_RN_pres ~ 1 +' 
    + ' + '.join(annees_dummies)
    + ' + EntityEffects'
)

Estimation du modèle

In [45]:
model_naif_typologie = PanelOLS.from_formula(formula_typologie, data=df_typologie, drop_absorbed=True, check_rank=False)
res_model_naif_typologie = model_naif_typologie.fit(cov_type='clustered', cluster_entity=True)

print("=== ÉVOLUTION DU VOTE RN : FOCUS RURAL NON PÉRIURBAIN (Réf: 1995) ===")
print(res_model_naif_typologie.summary)

c:\Users\yancr\anaconda3\envs\notebook\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


=== ÉVOLUTION DU VOTE RN : FOCUS RURAL NON PÉRIURBAIN (Réf: 1995) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:           vote_RN_pres   R-squared:                        0.6741
Estimator:                   PanelOLS   R-squared (Between):              0.0007
No. Observations:               96944   R-squared (Within):               0.6741
Date:                Wed, Apr 22 2026   R-squared (Overall):              0.3903
Time:                        16:31:10   Log-likelihood                 1.643e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   3.341e+04
Entities:                       16169   P-value                           0.0000
Avg Obs:                       5.9957   Distribution:                 F(5,80770)
Min Obs:                       3.0000                                           
Max Obs:                       6.0000  

#### Modèle avec controles

On filtre les données sur la typologie de communes

In [38]:
df_typologie = data[data['typologie_urbaine'] == 'Rural non périurbain'].copy()


Création des intéractions sur les années d'élections

In [39]:
# On crée une colonne pour chaque année d'élection sauf la première
annees_dummies = []
for annee in sorted(df_typologie['Annee'].unique()):
    if annee in [2002, 2007, 2012, 2017, 2022] :
        col_name = f'annee_{annee}'
        df_typologie[col_name] = (df_typologie['Annee'] == annee).astype(int)
        annees_dummies.append(col_name)

Création de la formule du modèle

In [40]:
# On définie les controles
controls = ['pop', 'propf', 'prop1539', 'prop4059', 'prop60p', 'petranger', 'pchom', 'pouem', 'paind', 'pbac', 'psup', 'revmoy']

# Note : On ne met PAS de 'TimeEffects' ici car les dummies années jouent exactement ce rôle.
# On garde les 'EntityEffects' pour voir l'évolution interne à chaque commune.
df_typologie = df_typologie.set_index(['codecommune', 'Annee'])

formula_typologie = (
    'vote_RN_pres ~ 1 +' 
    + ' + '.join(annees_dummies) + ' + ' 
    + ' + '.join(controls) 
    + ' + EntityEffects'
)

Estimation du modèle

In [41]:
model_controles_typologie = PanelOLS.from_formula(formula_typologie, data=df_typologie, drop_absorbed=True, check_rank=False)
res_model_controles_typologie = model_controles_typologie.fit(cov_type='clustered', cluster_entity=True)

print("=== ÉVOLUTION DU VOTE RN : FOCUS RURAL NON PÉRIURBAIN AVEC CONTROLES (Réf: 1995) ===")
print(res_model_controles_typologie.summary)

c:\Users\yancr\anaconda3\envs\notebook\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


=== ÉVOLUTION DU VOTE RN : FOCUS RURAL NON PÉRIURBAIN AVEC CONTROLES (Réf: 1995) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:           vote_RN_pres   R-squared:                        0.6900
Estimator:                   PanelOLS   R-squared (Between):              0.0088
No. Observations:               94789   R-squared (Within):               0.6900
Date:                Wed, Apr 22 2026   R-squared (Overall):              0.4033
Time:                        16:30:47   Log-likelihood                 1.636e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                    1.03e+04
Entities:                       16095   P-value                           0.0000
Avg Obs:                       5.8893   Distribution:                F(17,78677)
Min Obs:                       1.0000                                           
Max Obs:                

## Troisème Partie : Bouba Olga Pure

In [51]:
# 1. Sélection de l'année 2022
df_2022 = data[data['Annee'] == 2022].copy()

# 2. Création des Dummies de typologie
# On crée les colonnes et on les nettoie immédiatement (pas d'espaces, pas d'accents)
typo_dummies = pd.get_dummies(df_2022['typologie_urbaine'])
typo_dummies.columns = [c.replace(' ', '_').replace('é', 'e').replace('ê', 'e').replace('-', '_') for c in typo_dummies.columns]

# On fusionne avec le dataframe principal
df_2022 = pd.concat([df_2022, typo_dummies], axis=1)

# 3. On définit la référence (Urbain dense) et les autres catégories
typos_etudiees = [c for c in typo_dummies.columns if c != 'Urbain_dense']

# 4. Liste des contrôles socio-économiques
controls = ['pop', 'propf', 'prop1539', 'prop4059', 'prop60p', 'petranger', 'pchom', 'pouem', 'paind', 'pbac', 'psup', 'revmoy']

# ---------------------------------------------------------
# MODÈLE 1 : L'effet "Brut" du territoire
# ---------------------------------------------------------
formula_brut = 'vote_RN_pres ~ 1 + ' + ' + '.join(typos_etudiees)
res_brut = sm.OLS.from_formula(formula_brut, data=df_2022).fit(cov_type='HC3')

# ---------------------------------------------------------
# MODÈLE 2 : L'effet "Bouba-Olga" (Territoire + Composition Sociale)
# ---------------------------------------------------------
formula_complet = 'vote_RN_pres ~ 1 + ' + ' + '.join(typos_etudiees) + ' + ' + ' + '.join(controls)
res_complet = sm.OLS.from_formula(formula_complet, data=df_2022).fit(cov_type='HC3')

print("=== MODÈLE 1 : ÉCARTS TERRITORIAUX BRUTS (2022) ===")
print(res_brut.summary())
print("\n=== MODÈLE 2 : ÉCARTS TERRITORIAUX APRÈS CONTRÔLES (BOUBA-OLGA) ===")
print(res_complet.summary())

=== MODÈLE 1 : ÉCARTS TERRITORIAUX BRUTS (2022) ===
                            OLS Regression Results                            
Dep. Variable:           vote_RN_pres   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     581.6
Date:                Wed, 22 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:39:36   Log-Likelihood:                 33925.
No. Observations:               34827   AIC:                        -6.784e+04
Df Residuals:                   34823   BIC:                        -6.781e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------